In [ ]:
# what is collaborative filtering ;
# idea to implement 
# Find users who rate movies similarly to you, then recommend what they liked that you haven't seen yet
# two types user based and item based ; item based is generally better 

In [ ]:
#cosine similarity 
#Movie A vector ↔ Movie B vector
#User A vector ↔ User B vector
#pearson correlation

In [ ]:
#first KNN basic predictions 
#predict how user would rate a movie they havent seen yet 
#K most similar users who have rated that movie
#and we will multiply the rating given by them with the similarity score with you

In [ ]:
#KNNwithMeans 
#it a improved version of the knn basic predictions like person correlation is to cosine similarity 

In [3]:
import pandas as pd
from surprise import Dataset, Reader, KNNBasic, KNNWithMeans, accuracy

In [5]:
ratings=pd.read_csv("/Users/harshverma/Downloads/machine learning /movie recommendation system/ml-latest-small/ratings.csv")
ratings_sorted = ratings.sort_values('timestamp').reset_index(drop=True)
split_index = int(len(ratings_sorted) * 0.8)
train_df = ratings_sorted.iloc[:split_index]
test_df = ratings_sorted.iloc[split_index:]
print(train_df.shape,test_df.shape)

(80668, 4) (20168, 4)


In [6]:
reader = Reader(rating_scale=(0.5, 5.0))# reader library handles itself everything 
train_data = Dataset.load_from_df(train_df[['userId', 'movieId', 'rating']], reader)
# Pandas DataFrame
# Dataset.load_from_df()
# Surprise Dataset
trainset = train_data.build_full_trainset()
testset = list(zip(test_df['userId'], test_df['movieId'], test_df['rating']))
#in test set we didnt do the same as train bcz we just want to compare the values at last and in train we want to 
#prepare the data for the surprise to use 
print("Trainset size:", trainset.n_ratings)
print("Testset size:", len(testset))

Trainset size: 80668
Testset size: 20168


In [8]:
# item-based basic KNN
# Movie A → [ratings from users]
# Movie B → [ratings from users]
#        cosine similarity
#         similarity score
sim_options_item = {
    'name': 'cosine',
    'user_based': False   # item-based
}
algo_item = KNNBasic(sim_options=sim_options_item)
algo_item.fit(trainset)
predictions_item = algo_item.test(testset)
accuracy.rmse(predictions_item)
accuracy.mae(predictions_item)

Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 1.0769
MAE:  0.8495


np.float64(0.8495340947141022)

In [9]:
# used based basic knn
sim_options_user = {
    'name': 'pearson',
    'user_based': True   # user-based
}

algo_user = KNNBasic(sim_options=sim_options_user)
algo_user.fit(trainset)

predictions_user = algo_user.test(testset)

print("User-Based CF (Pearson):")
accuracy.rmse(predictions_user)
accuracy.mae(predictions_user)

Computing the pearson similarity matrix...
Done computing similarity matrix.
User-Based CF (Pearson):
RMSE: 1.0758
MAE:  0.8511


np.float64(0.851095977067569)

In [10]:
sim_options_means = {
    'name': 'cosine',
    'user_based': False
}

algo_means = KNNWithMeans(sim_options=sim_options_means)
algo_means.fit(trainset)

predictions_means = algo_means.test(testset)

print("Item-Based KNNWithMeans:")
accuracy.rmse(predictions_means)
accuracy.mae(predictions_means)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Item-Based KNNWithMeans:
RMSE: 1.0689
MAE:  0.8438


np.float64(0.8438216391587524)

In [ ]:
# CF underperformed mainly because the dataset is highly sparse, so KNN often
# had too few overlapping ratings to find reliable user/item similarities.
# The temporal split made training even sparser, and the default k=40 could
# include unreliable neighbors. Matrix Factorization should handle this better.